# RAG over the GDPR: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Regulation (EU) 2016/679, the *General Data Protection Regulation* (GDPR), official English PDF (88 pages).
It contains 173 recitals (explanatory text) followed by 99 articles (the actual law).

**Two pipelines over the same chunked corpus:**

| | Stage 1 | Stage 2 | Sent to LLM |
|---|---|---|---|
| **Pipeline 1 – Baseline** | Bi-Encoder retrieves Top 10 by cosine similarity | – | Top 3 by cosine similarity → **Answer 1** |
| **Pipeline 2 – Re-Ranking** | Same Top 10 candidates | Cross-Encoder scores each (query, chunk) pair and re-sorts | Top 3 by Cross-Encoder score → **Answer 2** |

Both pipelines use the identical prompt template and LLM settings, so any difference in the answers comes only from *which chunks* were retrieved.

**Notebook outline**
1. Setup & configuration
2. Load and clean the PDF
3. Structure-aware chunking
4. Bi-Encoder embeddings + FAISS vector index
5. LLM + shared prompt template
6. Pipeline 1 (baseline) and Pipeline 2 (re-ranking)
7. Sanity check on a sample query
8. Failure mode analysis (stress testing): rank-shift summary, detailed Answer 1 vs Answer 2, a counter-example and a limitation
9. Latency profiling

## 1. Setup & configuration
All tunable settings are kept in one place. The Gemini API key is read from a local `.env` file (`GEMINI_API_KEY=...`), which is **not** committed to Git.

In [1]:
import os
import re
import json
import time
import hashlib
import textwrap

import numpy as np
import pandas as pd
import pymupdf                   # PyMuPDF – PDF text extraction
import faiss                     # vector index
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

load_dotenv(".env")              # loads GEMINI_API_KEY from .env

# ---- Data ----
PDF_PATH = "data/gdpr.pdf"

# ---- Models ----
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # dense embedding model (Bi-Encoder)
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker (Cross-Encoder)
LLM_NAME           = "gemini-3.5-flash"                              # Gemini model used for answer generation

# ---- Retrieval ----
TOP_K = 10        # candidates retrieved by the Bi-Encoder
TOP_N = 3         # chunks passed to the LLM

# ---- Chunking ----
MAX_CHUNK_WORDS = 150   # ~200 word-piece tokens, safely under MiniLM's 256-token limit
OVERLAP_WORDS   = 30    # overlap used only when a single long paragraph has to be split

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## 2. Load and clean the PDF
The Official Journal layout adds noise that would pollute the chunks:
* a **running header** on every page (`4.5.2016 EN Official Journal of the European Union L 119/1`)
* **footnotes** (references to other Official Journal issues), set in a smaller font

PyMuPDF gives the position and font size of every text line, so:
* lines in the top header band are dropped;
* small-font (8.5 pt) lines are dropped as footnotes, **except** all-uppercase ones, because headings like *CHAPTER II* and *HAVE ADOPTED THIS REGULATION* also use the small font (as small capitals).

The PDF's text blocks are single lines, so paragraphs are rebuilt from the layout: a vertical gap larger than `PARA_GAP` between two lines starts a new paragraph.
At a page break, a paragraph continues onto the next page unless the previous line ended a sentence or the new line starts a numbered item or heading.
Soft hyphens (`\xad`) used to break words across lines are removed.

In [2]:
doc = pymupdf.open(PDF_PATH)
print(f"Pages: {doc.page_count}")

HEADER_BAND   = 65    # points from the top of the page occupied by the running header
BODY_MIN_FONT = 9.0   # body text is 9.6 pt; footnotes are 8.5 pt
PARA_GAP      = 5.0   # vertical gap (pt) between lines that signals a new paragraph

# lines that always begin a new paragraph: Article/CHAPTER/Section headings, recital "(12)", "3.", "(b)"
STARTS_NEW = re.compile(r"^(Article \d+$|CHAPTER [IVXL]+$|Section \d+$|\(\d+\)|\d+\.(\s|$)|\([a-z]\)\s)")

def page_lines(page):
    # Yield (text, top, bottom) for the body-text lines of one page.
    for block in page.get_text("dict", sort=True)["blocks"]:
        for line in block.get("lines", []):
            text = "".join(span["text"] for span in line["spans"]).strip()
            if not text or line["bbox"][1] < HEADER_BAND:                    # empty / running header
                continue
            small = max(span["size"] for span in line["spans"]) < BODY_MIN_FONT
            if small and text != text.upper():                               # footnote
                continue
            yield text, line["bbox"][1], line["bbox"][3]

def join_line(para, text):
    # Append a wrapped line to a paragraph, repairing words split across the line break.
    if para.endswith("\xad"):                 # soft hyphen: the word continues on the next line
        return para[:-1] + text
    if para.endswith("-"):                    # real hyphen (e.g. 'decision-' + 'making')
        return para + text
    return para + " " + text

def extract_paragraphs(doc):
    # Return a list of {'page', 'text'} paragraphs with headers and footnotes removed.
    paragraphs = []
    for page_no, page in enumerate(doc, start=1):
        prev_bottom = None                    # None = first line of this page
        for text, top, bottom in page_lines(page):
            if not paragraphs:
                new_para = True
            elif prev_bottom is None:         # page break: continue only a sentence cut mid-way
                last = paragraphs[-1]["text"]
                new_para = bool(STARTS_NEW.match(text)) or last.endswith((".", ":")) \
                           or bool(STARTS_NEW.match(last))
            else:
                new_para = (top - prev_bottom > PARA_GAP) or bool(STARTS_NEW.match(text))
            if new_para:
                paragraphs.append({"page": page_no, "text": text})
            else:
                paragraphs[-1]["text"] = join_line(paragraphs[-1]["text"], text)
            prev_bottom = bottom
    for p in paragraphs:
        p["text"] = re.sub(r"\s+", " ", p["text"].replace("\xad", "")).strip()
    return paragraphs

paragraphs = extract_paragraphs(doc)
print(f"Paragraphs extracted: {len(paragraphs)}")
for p in paragraphs[:6]:
    print(f"[p.{p['page']}] {p['text'][:150]}")

Pages: 88
Paragraphs extracted: 1256
[p.1] I
[p.1] (Legislative acts)
[p.1] REGULATIONS
[p.1] REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
[p.1] of 27 April 2016
[p.1] on the protection of natural persons with regard to the processing of personal data and on the free movement of such data, and repealing Directive 95/


## 3. Structure-aware chunking
Rather than cutting the text blindly every *N* words, we follow the document's own structure so every chunk can be traced back to a citable unit:

* **Recitals**, numbered `(1)` to `(173)`, between the opening formula and *"HAVE ADOPTED THIS REGULATION"*
* **Articles**, `Article 1` to `Article 99`, each followed by its title, up to the closing *"Done at Brussels"*
* `CHAPTER` and `Section` headings (and their titles) are structural only, so they are skipped

Inside a recital or article, consecutive paragraphs are **packed** into one chunk until `MAX_CHUNK_WORDS` would be exceeded.
A chunk never crosses an article or recital boundary. A single paragraph that is longer than the limit is split with a sliding word window (`OVERLAP_WORDS` overlap).

Each chunk carries metadata: `chunk_id`, `label` (e.g. *Article 9 – Processing of special categories of personal data*), and `page`.
The **same chunk list** is used by both pipelines.

In [3]:
RECITAL_START = re.compile(r"^\((\d{1,3})\)\s*(.*)$")
ARTICLE_HEAD  = re.compile(r"^Article (\d{1,2})$")
STRUCT_HEAD   = re.compile(r"^(CHAPTER [IVXL]+|Section \d+)$")

def split_into_sections(paragraphs):
    # Group paragraphs into recitals and articles: [{'label', 'kind', 'paras': [(page, text), ...]}]
    sections, current = [], None
    part = "preamble"            # preamble -> recitals -> articles -> end
    skip_next = False            # skip the title line that follows a CHAPTER/Section heading
    expect_title = False         # the line after "Article N" is the article's title

    for p in paragraphs:
        text, page = p["text"], p["page"]

        if part == "preamble":
            if RECITAL_START.match(text) and RECITAL_START.match(text).group(1) == "1":
                part = "recitals"
            else:
                continue

        if part == "recitals":
            if text.startswith("HAVE ADOPTED THIS REGULATION"):
                part = "articles"
                continue
            m = RECITAL_START.match(text)
            if m:
                current = {"label": f"Recital {m.group(1)}", "kind": "recital", "paras": []}
                sections.append(current)
                text = m.group(2)
                if not text:
                    continue
            current["paras"].append((page, text))
            continue

        if part == "articles":
            if text.startswith("Done at Brussels"):
                break
            if STRUCT_HEAD.match(text):
                skip_next = True
                continue
            if skip_next:
                skip_next = False
                continue
            m = ARTICLE_HEAD.match(text)
            if m:
                current = {"label": f"Article {m.group(1)}", "kind": "article", "paras": []}
                sections.append(current)
                expect_title = True
                continue
            if expect_title:
                current["label"] += f" – {text}"
                expect_title = False
                continue
            if current is not None:
                current["paras"].append((page, text))
    return sections


def chunk_section(section):
    # Pack a section's paragraphs into chunks of at most MAX_CHUNK_WORDS words.
    pieces = []   # (page, text) pieces, none longer than MAX_CHUNK_WORDS
    for page, text in section["paras"]:
        words = text.split()
        if len(words) <= MAX_CHUNK_WORDS:
            pieces.append((page, text))
        else:     # very long paragraph -> sliding window
            step = MAX_CHUNK_WORDS - OVERLAP_WORDS
            for start in range(0, len(words), step):
                pieces.append((page, " ".join(words[start:start + MAX_CHUNK_WORDS])))
                if start + MAX_CHUNK_WORDS >= len(words):
                    break

    chunks, buf, buf_page = [], [], None
    for page, text in pieces:
        if buf and sum(len(t.split()) for t in buf) + len(text.split()) > MAX_CHUNK_WORDS:
            chunks.append((buf_page, "\n".join(buf)))
            buf, buf_page = [], None
        if buf_page is None:
            buf_page = page
        buf.append(text)
    if buf:
        chunks.append((buf_page, "\n".join(buf)))
    return chunks


sections = split_into_sections(paragraphs)
chunks = []
for sec in sections:
    for page, text in chunk_section(sec):
        chunks.append({"chunk_id": len(chunks), "label": sec["label"], "kind": sec["kind"],
                       "page": page, "text": text})

chunk_df = pd.DataFrame(chunks)
chunk_df["words"] = chunk_df["text"].str.split().str.len()
n_recitals = chunk_df.loc[chunk_df.kind == "recital", "label"].nunique()
n_articles = chunk_df.loc[chunk_df.kind == "article", "label"].nunique()
print(f"Sections found: {n_recitals} recitals, {n_articles} articles")
print(f"Total chunks:   {len(chunks)}")
print(chunk_df["words"].describe().round(1).to_string())

Sections found: 173 recitals, 99 articles
Total chunks:   527
count    527.0
mean     104.3
std       38.1
min       22.0
25%       72.5
50%      112.0
75%      139.5
max      150.0


In [4]:
# Peek at a few chunks, including the famous Article 9 prohibition + exceptions
chunk_df[chunk_df.label.str.startswith("Article 9 ")][["chunk_id", "label", "page", "words", "text"]]

,chunk_id,label,page,words,text
282,282,Article 9 – Processing of special categories of personal data,38,111,"1. Processing of personal data revealing racial or ethnic origin, political opinions, ..."
283,283,Article 9 – Processing of special categories of personal data,38,103,(b) processing is necessary for the purposes of carrying out the obligations and exerc...
284,284,Article 9 – Processing of special categories of personal data,38,120,(d) processing is carried out in the course of its legitimate activities with appropri...
285,285,Article 9 – Processing of special categories of personal data,38,128,"(g) processing is necessary for reasons of substantial public interest, on the basis o..."
286,286,Article 9 – Processing of special categories of personal data,38,139,(i) processing is necessary for reasons of public interest in the area of public healt...
287,287,Article 9 – Processing of special categories of personal data,39,104,3. Personal data referred to in paragraph 1 may be processed for the purposes referred...


## 4. Bi-Encoder embeddings + FAISS vector index
The Bi-Encoder maps every chunk to a 384-dimensional vector **independently of any query**, so the whole corpus can be embedded once, ahead of time.
The vectors are L2-normalised, so the inner product computed by `faiss.IndexFlatIP` equals **cosine similarity**.
`IndexFlatIP` is an exact (brute-force) index, which is easily fast enough for a few hundred chunks.

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
print("Bi-Encoder max sequence length:", bi_encoder.max_seq_length, "tokens")

# Check that no chunk is truncated by the embedding model
tok_lens = [len(bi_encoder.tokenizer(c["text"])["input_ids"]) for c in chunks]
print(f"Chunk length in tokens: max={max(tok_lens)}, mean={np.mean(tok_lens):.0f}, "
      f"truncated chunks={sum(t > bi_encoder.max_seq_length for t in tok_lens)}")

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode([c["text"] for c in chunks], batch_size=64,
                                     normalize_embeddings=True, show_progress_bar=False)
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype("float32"))
print(f"Indexed {index.ntotal} chunks (dim={chunk_embeddings.shape[1]}) in {index_time:.1f}s (one-off cost)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Bi-Encoder max sequence length: 256 tokens
Chunk length in tokens: max=197, mean=122, truncated chunks=0


Indexed 527 chunks (dim=384) in 4.0s (one-off cost)


## 5. LLM + shared prompt template
Both pipelines call the **same** LLM with the **same** prompt template and `temperature=0`, so the answers are as deterministic and comparable as possible.
The prompt tells the model to answer only from the supplied context. That makes it visible when a pipeline failed to retrieve the right chunk: the model either says the information is missing or answers from the wrong passage.

In [6]:
llm_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PROMPT_TEMPLATE = '''You are an assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context passages below. Do not use outside knowledge.
If the context does not contain the information needed, say so explicitly.
Cite the passage labels you relied on, e.g. [Article 9].

Context:
{context}

Question: {question}

Answer:'''

# The free Gemini tier allows only a few requests per day, so every response is cached on disk.
# Re-running the notebook then reuses the saved answers (and their original latency)
# instead of calling the API again. Delete llm_cache.json to force fresh calls.
LLM_CACHE_PATH = "llm_cache.json"
try:
    with open(LLM_CACHE_PATH) as f:
        llm_cache = json.load(f)
except FileNotFoundError:
    llm_cache = {}

def build_context(chunk_ids):
    # Format the selected chunks as labelled passages for the prompt.
    return "\n\n".join(f"[{chunks[i]['label']}, page {chunks[i]['page']}]\n{chunks[i]['text']}"
                       for i in chunk_ids)

def generate_answer(question, chunk_ids, max_retries=4):
    # Call the LLM with the shared prompt. Returns (answer, seconds); uses the cache when possible.
    prompt = PROMPT_TEMPLATE.format(context=build_context(chunk_ids), question=question)
    key = hashlib.sha256(f"{LLM_NAME}\n{prompt}".encode()).hexdigest()
    if key in llm_cache:
        return llm_cache[key]["answer"], llm_cache[key]["seconds"]

    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            response = llm_client.models.generate_content(
                model=LLM_NAME, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            seconds = time.perf_counter() - t0
            break
        except Exception as e:
            if "PerDay" in str(e):                   # daily free-tier quota used up: retrying won't help
                raise RuntimeError("Daily Gemini quota exhausted; re-run later (cached answers are kept).") from e
            wait = 20 * (attempt + 1)                # per-minute rate limit / transient error
            print(f"  LLM call failed ({type(e).__name__}), retrying in {wait}s")
            time.sleep(wait)
    else:
        raise RuntimeError("LLM call failed after retries")

    answer = response.text.strip()
    llm_cache[key] = {"model": LLM_NAME, "question": question, "chunk_ids": list(chunk_ids),
                      "answer": answer, "seconds": seconds}
    with open(LLM_CACHE_PATH, "w") as f:
        json.dump(llm_cache, f, indent=1, ensure_ascii=False)
    return answer, seconds

## 6. The two retrieval pipelines

**Pipeline 1 (Baseline):** embed the query → FAISS search for the Top 10 by cosine similarity → send the Top 3 to the LLM → **Answer 1**.

**Pipeline 2 (Re-Ranking):** take the *same* Top 10 → build 10 `(query, chunk)` pairs → the Cross-Encoder reads each pair *jointly* and outputs a relevance score → re-sort descending → send the new Top 3 to the LLM → **Answer 2**.

Every stage is timed separately with `time.perf_counter()`.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")

def bi_encoder_retrieve(query, k=TOP_K):
    # Stage 1: dense retrieval. Returns (chunk_ids, cosine_scores, seconds).
    t0 = time.perf_counter()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return ids[0].tolist(), scores[0].tolist(), time.perf_counter() - t0

def cross_encoder_rerank(query, candidate_ids):
    # Stage 2: score every (query, chunk) pair jointly and re-sort. Returns (ids, scores, seconds).
    t0 = time.perf_counter()
    pairs = [(query, chunks[i]["text"]) for i in candidate_ids]
    ce_scores = cross_encoder.predict(pairs)
    order = np.argsort(-ce_scores)
    return [candidate_ids[j] for j in order], [float(ce_scores[j]) for j in order], time.perf_counter() - t0

def run_pipelines(query, with_llm=True):
    # Run both pipelines on one query and collect ranks, answers and timings.
    bi_ids, bi_scores, t_retrieve = bi_encoder_retrieve(query)
    ce_ids, ce_scores, t_rerank = cross_encoder_rerank(query, bi_ids)

    # One row per candidate, showing its rank under both orderings
    ce_rank  = {cid: r for r, cid in enumerate(ce_ids, start=1)}
    ce_score = dict(zip(ce_ids, ce_scores))
    rows = []
    for r, (cid, s) in enumerate(zip(bi_ids, bi_scores), start=1):
        rows.append({"chunk_id": cid, "label": chunks[cid]["label"][:60], "page": chunks[cid]["page"],
                     "bi_rank": r, "cosine": round(s, 3),
                     "ce_rank": ce_rank[cid], "ce_score": round(ce_score[cid], 2),
                     "shift": r - ce_rank[cid]})
    ranks = pd.DataFrame(rows)

    result = {"query": query, "ranks": ranks,
              "baseline_top": bi_ids[:TOP_N], "rerank_top": ce_ids[:TOP_N],
              "t_retrieve": t_retrieve, "t_rerank": t_rerank}

    if with_llm:
        result["answer1"], result["t_llm1"] = generate_answer(query, bi_ids[:TOP_N])   # Pipeline 1
        result["answer2"], result["t_llm2"] = generate_answer(query, ce_ids[:TOP_N])   # Pipeline 2
    return result

def short(cid):
    # 'Article 17 – Right to erasure ...' -> 'Article 17 (#311)'
    return f"{chunks[cid]['label'].split(' –')[0]} (#{cid})"

def show_result(res, gold_ids=()):
    # Pretty-print the rank table and both answers for one query. Gold chunks are marked with ★.
    print("QUERY:", res["query"])
    table = res["ranks"].copy()
    table.insert(0, "gold", ["★" if c in gold_ids else "" for c in table.chunk_id])
    print(f"\nTop {TOP_K} candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):")
    display(table.style.hide(axis="index")
            .apply(lambda row: ["background-color: rgba(46,160,67,0.18)" if row.ce_rank <= TOP_N else ""] * len(row), axis=1)
            .format({"cosine": "{:.3f}", "ce_score": "{:.2f}"}))
    print("Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).")
    print(f"\nBaseline Top {TOP_N} : {[short(i) for i in res['baseline_top']]}")
    print(f"Re-ranked Top {TOP_N}: {[short(i) for i in res['rerank_top']]}")
    if "answer1" in res:
        w = lambda s: "\n".join(textwrap.fill(line, 110) for line in s.splitlines())
        print("\n" + "-" * 110 + "\nANSWER 1 (Baseline):\n" + w(res["answer1"]))
        print("\n" + "-" * 110 + "\nANSWER 2 (Re-ranked):\n" + w(res["answer2"]))
    print("\n" + "-" * 110)
    print(f"Retrieval {res['t_retrieve']*1000:.1f} ms | Re-rank {res['t_rerank']*1000:.1f} ms"
          + (f" | LLM {res['t_llm1']:.2f}s / {res['t_llm2']:.2f}s" if "t_llm1" in res else ""))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 7. Sanity check on a sample query
A straightforward factual question, to confirm the whole pipeline works end to end. Here both pipelines should agree.

In [8]:
res = run_pipelines("How quickly must a personal data breach be reported to the supervisory authority?")
show_result(res)

QUERY: How quickly must a personal data breach be reported to the supervisory authority?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,131,Recital 87,17,1,0.744,3,5.07,-2
,129,Recital 85,17,2,0.728,1,7.87,1
,350,Article 33 – Notification of a personal data breach to the s,52,3,0.727,2,7.39,1
,351,Article 33 – Notification of a personal data breach to the s,52,4,0.670,6,2.28,-2
,354,Article 34 – Communication of a personal data breach to the,53,5,0.642,5,3.62,0
,130,Recital 86,17,6,0.639,4,4.23,2
,474,Article 70 – Tasks of the Board,77,7,0.620,10,-5.37,-3
,132,Recital 88,17,8,0.618,9,-1.44,-1
,352,Article 34 – Communication of a personal data breach to the,52,9,0.618,8,-0.22,1
,128,Recital 85,16,10,0.596,7,-0.18,3


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Recital 87 (#131)', 'Recital 85 (#129)', 'Article 33 (#350)']
Re-ranked Top 3: ['Recital 85 (#129)', 'Article 33 (#350)', 'Recital 87 (#131)']

--------------------------------------------------------------------------------------------------------------
ANSWER 1 (Baseline):
Based on the provided context, a controller must notify a personal data breach to the supervisory authority
**without undue delay** and, where feasible, **not later than 72 hours** after becoming aware of it [Recital
85, page 17; Article 33, page 52].

Key details regarding this timeframe include:
* **Exception:** Notification is not required if the personal data breach is unlikely to result in a risk to
the rights and freedoms of natural persons [Recital 85, page 17; Article 33, page 52].
* **Delays:** If the notification is not made within 72 hours, it must be accompanied by reasons for the delay
[Rec

## 8. Failure mode analysis (stress testing)

**Method**
1. Write a question, then locate its **gold chunk** (the chunk that actually contains the answer) by hand, using a regex on a distinctive phrase from the legal text.
2. Find the gold chunk's rank in the **baseline** ordering (over the whole corpus, so we also see when it falls outside the Top 10) and in the **re-ranked** Top 10.
3. Questions where the gold chunk sits at baseline rank 4–10 and is promoted into the Top 3 are the cases where re-ranking should change the LLM's answer. These are examined in detail in 8.2.

The questions are deliberately phrased the way a normal person would ask them ("fired", "delete", "small violation"), while the regulation uses legal terms ("dismissed", "erasure", "minor infringement"). This vocabulary gap is where a Bi-Encoder struggles.

In [9]:
STRESS_TESTS = [
    # --- cases where re-ranking should help (examined in detail in 8.2) ---
    dict(type="Keyword trap / distractor",
         query="Can a supervisory authority just give a warning instead of a fine for a small violation?",
         gold=r"minor infringement",
         note="Recital 150 literally contains 'giving a warning'; the actual rule (Recital 148) says 'minor infringement' and 'reprimand'"),
    dict(type="Keyword trap / synonym",
         query="Can I ask a company to delete my data?",
         gold=r"right to obtain from the controller the erasure",
         note="'delete' vs the legal term 'erasure' (Article 17(1))"),
    dict(type="Exception / corner case",
         query="Can a country limit my data protection rights for reasons of national security?",
         gold=r"may restrict by way of a legislative measure the scope of the obligations",
         note="the rights are granted in Articles 12-22; the restriction is a separate article (Article 23)"),
    dict(type="Exception / corner case",
         query="Can the right to get a copy of my data be limited to protect other people?",
         gold=r"right to obtain a copy referred to in paragraph 3 shall not adversely affect",
         note="limitation sits in Article 15(4); Recital 68 (data portability, a different right) is a near-duplicate distractor"),
    dict(type="Negation & constraints",
         query="Can the data protection officer be fired for doing their job?",
         gold=r"shall not be dismissed or penalised",
         note="'fired' vs 'shall not be dismissed or penalised' (Article 38(3))"),
    dict(type="Buried chunk (rank 4-10)",
         query="Is pseudonymised data still considered personal data?",
         gold=r"which have undergone pseudonymisation",
         note="answer is in the first half of Recital 26; the definitions article and the anonymous-data half of Recital 26 outrank it"),
    # --- more cases, screened by rank only ---
    dict(type="Exception / corner case",
         query="Can I object to my data being used for direct marketing even if the company has a legitimate interest?",
         gold=r"processed for direct marketing purposes, the data subject shall have the right to object at any time",
         note="absolute right to object, Article 21(2)"),
    dict(type="Buried chunk (rank 4-10)",
         query="Can an organisation file a complaint on my behalf?",
         gold=r"right to mandate a not-for-profit body",
         note="Recital 142 / Article 80"),
    # --- re-ranking makes things worse ---
    dict(type="Counter-example",
         query="Is a processor liable for damages in the same way as the controller?",
         gold=r"processor shall be liable for the damage caused by processing only where",
         note="Article 82(2): baseline already has it at #1"),
    dict(type="Counter-example",
         query="When is a data protection impact assessment mandatory?",
         gold=r"shall in particular be required in the case of",
         note="Article 35(3)"),
    # --- gold chunk never reaches the Top 10: re-ranking cannot help ---
    dict(type="Limitation (negation)",
         query="Which types of personal data is a company not allowed to process?",
         gold=r"sexual orientation shall be prohibited",
         note="Article 9(1): the prohibition list"),
    dict(type="Limitation (negation)",
         query="Are courts supervised by data protection authorities?",
         gold=r"shall not be competent to supervise processing operations of courts",
         note="Article 55(3)"),
    dict(type="Limitation (keyword trap)",
         query="How much can a company be fined for breaking the basic principles for processing?",
         gold=r"basic principles for processing, including conditions for consent",
         note="Article 83(5): 20 M EUR / 4 %; the 10 M EUR / 2 % tier in Article 83(4) is the distractor"),
]

def gold_chunks(pattern):
    # Chunk ids whose text matches the gold regex.
    return [c["chunk_id"] for c in chunks if re.search(pattern, c["text"].replace("\n", " "), re.I)]

def full_baseline_rank(query, cid):
    # Rank of a chunk among ALL chunks by cosine similarity (1 = best).
    q = bi_encoder.encode([query], normalize_embeddings=True)[0]
    order = np.argsort(-(chunk_embeddings @ q))
    return int(np.where(order == cid)[0][0]) + 1

def outcome(b, r):
    if b > TOP_K:      return "not retrieved (outside Top 10)"
    if b <= TOP_N and r <= TOP_N: return "in Top 3 both times"
    if b > TOP_N and r <= TOP_N:  return "FIXED: promoted into Top 3"
    if b <= TOP_N and r > TOP_N:  return "HURT: pushed out of Top 3"
    return "missed both times"

rows = []
for t in STRESS_TESTS:
    res = run_pipelines(t["query"], with_llm=False)
    ce_rank = dict(zip(res["ranks"].chunk_id, res["ranks"].ce_rank))
    golds = gold_chunks(t["gold"])
    t["gold_ids"] = golds
    best = min(golds, key=lambda g: full_baseline_rank(t["query"], g))   # best-placed gold chunk
    b = full_baseline_rank(t["query"], best)
    r = ce_rank.get(best, None)
    rows.append({"type": t["type"], "query": t["query"], "gold chunk": short(best),
                 "baseline rank": b, "re-ranked rank": r if r else "-",
                 "rank shift": f"#{b} → #{r}" if r else f"#{b} → (not in Top 10)",
                 "outcome": outcome(b, r or 99)})

stress_summary = pd.DataFrame(rows)
stress_summary

,type,query,gold chunk,baseline rank,re-ranked rank,rank shift,outcome
0,Keyword trap / distractor,Can a supervisory authority just give a warning instead of a fine for a small violation?,Recital 148 (#220),4,1,#4 → #1,FIXED: promoted into Top 3
1,Keyword trap / synonym,Can I ask a company to delete my data?,Article 17 (#311),4,2,#4 → #2,FIXED: promoted into Top 3
2,Exception / corner case,Can a country limit my data protection rights for reasons of national security?,Article 23 (#324),9,2,#9 → #2,FIXED: promoted into Top 3
3,Exception / corner case,Can the right to get a copy of my data be limited to protect other people?,Article 15 (#309),5,1,#5 → #1,FIXED: promoted into Top 3
4,Negation & constraints,Can the data protection officer be fired for doing their job?,Article 38 (#368),8,3,#8 → #3,FIXED: promoted into Top 3
5,Buried chunk (rank 4-10),Is pseudonymised data still considered personal data?,Recital 26 (#32),9,3,#9 → #3,FIXED: promoted into Top 3
6,Exception / corner case,Can I object to my data being used for direct marketing even if the company has a legi...,Article 21 (#319),5,1,#5 → #1,FIXED: promoted into Top 3
7,Buried chunk (rank 4-10),Can an organisation file a complaint on my behalf?,Recital 142 (#208),6,2,#6 → #2,FIXED: promoted into Top 3
8,Counter-example,Is a processor liable for damages in the same way as the controller?,Article 82 (#495),1,4,#1 → #4,HURT: pushed out of Top 3
9,Counter-example,When is a data protection impact assessment mandatory?,Article 35 (#355),6,8,#6 → #8,missed both times


### 8.2 Detailed comparison: Answer 1 vs Answer 2
For the first six stress tests the gold chunk is outside the baseline Top 3 but inside the re-ranked Top 3. Both pipelines are now run with the LLM, using the identical prompt, so we can see what the missing chunk does to the generated answer.

In [10]:
detailed = {}
for t in STRESS_TESTS[:6]:
    print("=" * 110)
    print(f"[{t['type']}]  {t['note']}")
    detailed[t["query"]] = run_pipelines(t["query"])
    show_result(detailed[t["query"]], gold_ids=t["gold_ids"])
    print()

[Keyword trap / distractor]  Recital 150 literally contains 'giving a warning'; the actual rule (Recital 148) says 'minor infringement' and 'reprimand'
QUERY: Can a supervisory authority just give a warning instead of a fine for a small violation?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,221,Recital 148,27,1,0.679,6,-1.08,-5
,503,Article 83 – General conditions for imposing administrative,83,2,0.638,7,-1.40,-5
,225,Recital 150,28,3,0.636,2,1.55,1
★,220,Recital 148,27,4,0.615,1,4.04,3
,223,Recital 150,27,5,0.595,4,0.14,1
,498,Article 83 – General conditions for imposing administrative,82,6,0.594,3,0.66,3
,192,Recital 130,24,7,0.571,8,-1.72,-1
,504,Article 83 – General conditions for imposing administrative,83,8,0.563,5,-0.20,3
,502,Article 83 – General conditions for imposing administrative,83,9,0.560,9,-2.10,0
,190,Recital 129,24,10,0.559,10,-2.29,0


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Recital 148 (#221)', 'Article 83 (#503)', 'Recital 150 (#225)']
Re-ranked Top 3: ['Recital 148 (#220)', 'Recital 150 (#225)', 'Article 83 (#498)']

--------------------------------------------------------------------------------------------------------------
ANSWER 1 (Baseline):
Based on the provided context, it is not explicitly stated whether a supervisory authority can give a warning
instead of a fine for a small violation.

The context mentions that "giving a warning does not affect the application of other powers of the supervisory
authorities or of other penalties under this Regulation" [Recital 150, page 28], but it does not contain the
specific rules or conditions regarding the use of warnings for small violations.

--------------------------------------------------------------------------------------------------------------
ANSWER 2 (Re-ranked):
Yes. In the case of

QUERY: Can I ask a company to delete my data?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,312,Article 17 – Right to erasure (‘right to be forgotten’),44,1,0.525,4,-6.35,-3
,96,Recital 65,13,2,0.511,1,-3.07,1
,53,Recital 39,7,3,0.507,3,-5.78,0
★,311,Article 17 – Right to erasure (‘right to be forgotten’),43,4,0.493,2,-5.39,2
,97,Recital 66,13,5,0.487,5,-6.63,0
,95,Recital 65,12,6,0.483,6,-6.71,0
,101,Recital 68,13,7,0.478,7,-8.76,0
,209,Recital 142,26,8,0.457,8,-9.33,0
,151,Recital 102,19,9,0.454,10,-10.56,-1
,91,Recital 62,12,10,0.450,9,-9.96,1


...

gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,369,Article 38 – Position of the data protection officer,56,1,0.594,2,1.91,-1
,367,Article 37 – Designation of the data protection officer,55,2,0.588,1,2.44,1
,370,Article 39 – Tasks of the data protection officer,56,3,0.556,4,0.77,-1
,365,Article 37 – Designation of the data protection officer,55,4,0.517,7,-0.73,-3
,131,Recital 87,17,5,0.491,9,-5.80,-4
,354,Article 34 – Communication of a personal data breach to the,53,6,0.484,10,-5.87,-4
,366,Article 37 – Designation of the data protection officer,55,7,0.473,6,-0.57,1
★,368,Article 38 – Position of the data protection officer,55,8,0.472,3,1.88,5
,371,Article 39 – Tasks of the data protection officer,56,9,0.461,5,0.50,4
,351,Article 33 – Notification of a personal data breach to the s,52,10,0.459,8,-2.25,2


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Article 38 (#369)', 'Article 37 (#367)', 'Article 39 (#370)']
Re-ranked Top 3: ['Article 37 (#367)', 'Article 38 (#369)', 'Article 38 (#368)']

--------------------------------------------------------------------------------------------------------------
ANSWER 1 (Baseline):
Based on the provided context, there is no information about whether a data protection officer can be fired or
dismissed for performing their job.

--------------------------------------------------------------------------------------------------------------
ANSWER 2 (Re-ranked):
No, the data protection officer cannot be fired (dismissed) or penalised by the controller or the processor
for performing their tasks [Article 38 – Position of the data protection officer, page 55].

--------------------------------------------------------------------------------------------------------------
Retrieval 5.4 ms 

QUERY: Is pseudonymised data still considered personal data?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,35,Recital 28,5,1,0.768,1,7.28,0
,33,Recital 26,5,2,0.682,5,3.05,-3
,260,Article 4 – Definitions,33,3,0.671,4,3.83,-1
,5,Recital 6,2,4,0.669,9,-2.03,-5
,277,Article 6 – Lawfulness of processing,37,5,0.669,6,2.42,-1
,36,Recital 29,5,6,0.661,2,5.58,4
,508,Article 86 – Processing and public access to official docume,84,7,0.636,8,0.58,-1
,91,Recital 62,12,8,0.621,10,-2.78,-2
★,32,Recital 26,5,9,0.610,3,4.21,6
,117,Recital 78,15,10,0.606,7,2.11,3


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Recital 28 (#35)', 'Recital 26 (#33)', 'Article 4 (#260)']
Re-ranked Top 3: ['Recital 28 (#35)', 'Recital 29 (#36)', 'Recital 26 (#32)']

--------------------------------------------------------------------------------------------------------------
ANSWER 1 (Baseline):
Based on the provided context, the passages do not explicitly state whether pseudonymised data is still
considered personal data.

The context notes that:
* "Pseudonymisation" is defined as "the processing of personal data in such a manner that the personal data
can no longer be attributed to a specific data subject without the use of additional information..." [[Article
4 – Definitions, page 33]].
* The application of pseudonymisation to personal data helps reduce risks to data subjects [[Recital 28, page
5]].
* The principles of data protection do not apply to anonymous information where the data subject is

### 8.3 Counter-example: re-ranking can also hurt
The Cross-Encoder (trained on MS MARCO web search queries) is not always right. Here the baseline already had the gold chunk at rank 1, and re-ranking pushes it out of the Top 3.
Only retrieval is shown here, because the daily free-tier LLM quota ran out. The rank table and the dropped clause are enough to see what Pipeline 2 would lose: the condition limiting a processor's liability is no longer in the context sent to the LLM.

In [11]:
t = STRESS_TESTS[8]
print(f"[{t['type']}]  {t['note']}")
res_counter = run_pipelines(t["query"], with_llm=False)
show_result(res_counter, gold_ids=t["gold_ids"])

gold = t["gold_ids"][0]
print(f"\nGold chunk {short(gold)}, in the baseline Top 3 but dropped by the re-ranker:")
print(textwrap.fill(chunks[gold]["text"].replace("\n", " "), 110))

[Counter-example]  Article 82(2): baseline already has it at #1


QUERY: Is a processor liable for damages in the same way as the controller?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
★,495,Article 82 – Right to compensation and liability,81,1,0.817,4,7.20,-3
,496,Article 82 – Right to compensation and liability,81,2,0.759,1,8.48,1
,217,Recital 146,27,3,0.730,2,8.17,1
,218,Recital 146,27,4,0.707,3,8.01,1
,339,Article 28 – Processor,50,5,0.601,5,5.14,0
,499,Article 83 – General conditions for imposing administrative,82,6,0.550,7,2.42,-1
,216,Recital 145,27,7,0.518,6,2.92,1
,334,Article 28 – Processor,49,8,0.518,8,-0.41,0
,368,Article 38 – Position of the data protection officer,55,9,0.514,9,-2.57,0
,341,Article 28 – Processor,50,10,0.513,10,-5.12,0


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Article 82 (#495)', 'Article 82 (#496)', 'Recital 146 (#217)']
Re-ranked Top 3: ['Article 82 (#496)', 'Recital 146 (#217)', 'Recital 146 (#218)']

--------------------------------------------------------------------------------------------------------------
Retrieval 5.5 ms | Re-rank 92.6 ms

Gold chunk Article 82 (#495), in the baseline Top 3 but dropped by the re-ranker:
1. Any person who has suffered material or non-material damage as a result of an infringement of this
Regulation shall have the right to receive compensation from the controller or processor for the damage
suffered. 2. Any controller involved in processing shall be liable for the damage caused by processing which
infringes this Regulation. A processor shall be liable for the damage caused by processing only where it has
not complied with obligations of this Regulation specifically directed to processors o

### 8.4 Limitation: re-ranking cannot recover a chunk the Bi-Encoder never retrieved
Re-ranking only re-orders the Top 10 candidates. If the gold chunk is not among them (Article 9(1) ranks far down for this negated question), the Cross-Encoder never sees it.

In [12]:
t = STRESS_TESTS[10]
res_lim = run_pipelines(t["query"], with_llm=False)
print(f"Gold chunk {short(t['gold_ids'][0])} baseline rank over the full corpus: "
      f"#{full_baseline_rank(t['query'], t['gold_ids'][0])}")
show_result(res_lim, gold_ids=t["gold_ids"])

Gold chunk Article 9 (#282) baseline rank over the full corpus: #86
QUERY: Which types of personal data is a company not allowed to process?

Top 10 candidates (bi_rank = baseline order, ce_rank = after re-ranking, shift = places moved up):


gold,chunk_id,label,page,bi_rank,cosine,ce_rank,ce_score,shift
,74,Recital 51,10,1,0.671,1,3.69,0
,54,Recital 40,7,2,0.663,4,0.99,-2
,38,Recital 31,6,3,0.655,6,-0.01,-3
,14,Recital 13,3,4,0.648,10,-1.87,-6
,52,Recital 39,7,5,0.646,5,0.54,0
,336,Article 28 – Processor,49,6,0.637,7,-0.05,-1
,287,Article 9 – Processing of special categories of personal dat,39,7,0.631,9,-1.25,-2
,315,Article 18 – Right to restriction of processing,44,8,0.621,3,1.66,5
,93,Recital 63,12,9,0.621,8,-1.03,1
,82,Recital 54,11,10,0.618,2,3.22,8


Green rows = the Top 3 the re-ranker sends to the LLM.  ★ = gold chunk (contains the answer).

Baseline Top 3 : ['Recital 51 (#74)', 'Recital 40 (#54)', 'Recital 31 (#38)']
Re-ranked Top 3: ['Recital 51 (#74)', 'Recital 54 (#82)', 'Article 18 (#315)']

--------------------------------------------------------------------------------------------------------------
Retrieval 5.3 ms | Re-rank 87.2 ms


## 9. Latency profiling
Each query is run `REPEATS` times after a warm-up call, and we report the mean time per stage (CPU only):
* **Baseline retrieval**: query embedding + FAISS search
* **Re-ranking**: Cross-Encoder scoring of the 10 candidates (on top of the baseline retrieval)
* **LLM generation**: measured separately, since it dominates end-to-end time but is the same work for both pipelines

To support the scaling question, we also time the Cross-Encoder over **every chunk in the corpus** instead of just the Top 10.

In [13]:
PROFILE_QUERIES = [t["query"] for t in STRESS_TESTS]
REPEATS = 5

# warm-up (first calls include one-off overheads)
_ids, _, _ = bi_encoder_retrieve(PROFILE_QUERIES[0]); cross_encoder_rerank(PROFILE_QUERIES[0], _ids)

t_retr, t_rr = [], []
for q in PROFILE_QUERIES:
    for _ in range(REPEATS):
        ids, _, tr = bi_encoder_retrieve(q)
        _, _, trr = cross_encoder_rerank(q, ids)
        t_retr.append(tr); t_rr.append(trr)
t_retr, t_rr = np.array(t_retr) * 1000, np.array(t_rr) * 1000

# Cross-Encoder over the entire corpus (what we would pay without the Bi-Encoder stage)
t_full = []
for q in PROFILE_QUERIES[:3]:
    t0 = time.perf_counter()
    cross_encoder.predict([(q, c["text"]) for c in chunks], batch_size=32)
    t_full.append((time.perf_counter() - t0) * 1000)

# LLM latency: all generation calls made in this notebook (originally measured, read from the cache)
llm_times = np.array([v["seconds"] for v in llm_cache.values() if v["model"] == LLM_NAME]) * 1000

latency = pd.DataFrame([
    {"stage": "Pipeline 1: Bi-Encoder retrieval (embed query + FAISS Top 10)",
     "mean_ms": t_retr.mean(), "std_ms": t_retr.std(), "runs": len(t_retr)},
    {"stage": "Cross-Encoder re-rank of Top 10",
     "mean_ms": t_rr.mean(), "std_ms": t_rr.std(), "runs": len(t_rr)},
    {"stage": "Pipeline 2: retrieval + re-rank",
     "mean_ms": (t_retr + t_rr).mean(), "std_ms": (t_retr + t_rr).std(), "runs": len(t_rr)},
    {"stage": f"Cross-Encoder over ALL {len(chunks)} chunks (no Bi-Encoder)",
     "mean_ms": np.mean(t_full), "std_ms": np.std(t_full), "runs": len(t_full)},
    {"stage": f"LLM generation, one call ({LLM_NAME})",
     "mean_ms": llm_times.mean(), "std_ms": llm_times.std(), "runs": len(llm_times)},
])
latency["vs_baseline"] = (latency.mean_ms / t_retr.mean()).round(1).astype(str) + "x"
print(f"{len(PROFILE_QUERIES)} queries x {REPEATS} repeats, CPU only")
latency.round(1)

13 queries x 5 repeats, CPU only


,stage,mean_ms,std_ms,runs,vs_baseline
0,Pipeline 1: Bi-Encoder retrieval (embed query + FAISS Top 10),6.8,1.0,65,1.0x
1,Cross-Encoder re-rank of Top 10,90.7,4.6,65,13.3x
2,Pipeline 2: retrieval + re-rank,97.6,4.8,65,14.3x
3,Cross-Encoder over ALL 527 chunks (no Bi-Encoder),3944.6,59.3,3,577.9x
4,"LLM generation, one call (gemini-3.5-flash)",5724.6,2031.8,15,838.7x


In [14]:
# End-to-end time per query (retrieval + LLM call), using the detailed runs from section 8.2
e2e = pd.DataFrame([{
    "query": q[:70],
    "pipeline 1 total (s)": r["t_retrieve"] + r["t_llm1"],
    "pipeline 2 total (s)": r["t_retrieve"] + r["t_rerank"] + r["t_llm2"],
    "re-rank share of pipeline 2": r["t_rerank"] / (r["t_retrieve"] + r["t_rerank"] + r["t_llm2"]),
} for q, r in detailed.items()])
e2e.round(3).style.format({"re-rank share of pipeline 2": "{:.1%}"}).hide(axis="index")

query,pipeline 1 total (s),pipeline 2 total (s),re-rank share of pipeline 2
Can a supervisory authority just give a warning instead of a fine for,8.875000,6.365000,1.5%
Can I ask a company to delete my data?,9.143000,6.923000,1.3%
Can a country limit my data protection rights for reasons of national,3.394000,5.423000,1.7%
Can the right to get a copy of my data be limited to protect other peo,5.795000,4.351000,2.1%
Can the data protection officer be fired for doing their job?,3.561000,3.032000,2.7%
Is pseudonymised data still considered personal data?,9.584000,4.516000,1.9%
